In [2]:
import time

import numpy as np
import pandas as pd

from Pipeline.Model.ABC_ELM2 import ABC_ELM2
from Pipeline.Model.DataSplit import DataSplit
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix

In [3]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]
features_size = X.shape[1]

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [5]:
splitter = DataSplit(random_state=42, test_size=0.2, k_fold=2)
x_test, y_test, folds = splitter.k_fold_data_spiting(X, y)

# We will demonstrate the metaheuristic optimization on Fold 0
fold_0 = folds[0]
x_train_f0 = fold_0['X_train_fold']
y_train_f0 = fold_0['y_train_fold']
x_val_f0   = fold_0['X_val_fold']
y_val_f0   = fold_0['y_val_fold']

print("\n[+] Initializing ABC(II)-ELM Optimization...")

# Engineering Reality: Cap iterations, enforce seeds, define the algorithm path.
abc_model = ABC_ELM2(
    feature_size=x_train_f0.shape[1],
    hidden_size=32,
    activation_function=sigmoid,
    regularization_lambda=1.223323,
    algo_type='algo3',   # Explicitly routing to Algorithm 3
    random_seed=42,      # Enforcing reproducibility
    SN=10,
    limit=15,
    iter_max=100         # Brought down to paper standard
)

# Fit the model
start_time = time.time()
abc_model.fit(x_train_f0, y_train_f0)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

# Evaluate
y_pred_abc = abc_model.predict(x_val_f0)
eval_metrics = EvaluationMatrix(y_val_f0, y_pred_abc)

print("\n=== ABC-ELM Optimization Results (Fold 0) ===")
for metric, value in eval_metrics.get_all_metrics().items():
    print(f"{metric}: {value:.4f}")


[+] Initializing ABC(II)-ELM Optimization...
Iteration 1 end : 0.2592s
Iteration 2 end : 0.3241s
Iteration 3 end : 0.2412s
Iteration 4 end : 0.2952s
Iteration 5 end : 0.2160s
Iteration 6 end : 0.2292s
Iteration 7 end : 0.2382s
Iteration 8 end : 0.2131s
Iteration 9 end : 0.2170s
Iteration 10 end : 0.2450s
Iteration 11 end : 0.2340s
Iteration 12 end : 0.2304s
Iteration 13 end : 0.2499s
Iteration 14 end : 0.2338s
Iteration 15 end : 0.2478s
Iteration 16 end : 0.1947s
Iteration 17 end : 0.1948s
Iteration 18 end : 0.4050s
Iteration 19 end : 0.2497s
Iteration 20 end : 0.3011s
Iteration 21 end : 0.2287s
Iteration 22 end : 0.2622s
Iteration 23 end : 0.2517s
Iteration 24 end : 0.2640s
Iteration 25 end : 0.2726s
Iteration 26 end : 0.2428s
Iteration 27 end : 0.2458s
Iteration 28 end : 0.2556s
Iteration 29 end : 0.2470s
Iteration 30 end : 0.2858s
Iteration 31 end : 0.2320s
Iteration 32 end : 0.2526s
Iteration 33 end : 0.2605s
Iteration 34 end : 0.2348s
Iteration 35 end : 0.2224s
Iteration 36 end :

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.special import expit

# Import the existing pipeline modules
from Pipeline.Model.DataSplit import DataSplit
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix
from Pipeline.Model.ABC_ELM2 import ABC_ELM2  # Assuming refactored version

def stable_sigmoid(x):
    # Numerically stable activation to prevent float64 overflow in randomized spaces
    return expit(x)

class AcademicEvaluation:
    @staticmethod
    def rigorous_metaheuristic_validation(X, y,
                                          k_folds=5,
                                          seeds_per_fold=30,
                                          hidden_size=32,
                                          reg_lambda=1.0,
                                          algo_type='algo3'):
        """
        Executes a nested cross-validation to isolate data variance from algorithmic variance.
        """
        print(f"[*] Initializing Rigorous Evaluation: {k_folds}-Fold CV with {seeds_per_fold} Seeds/Fold.")

        # 1. Isolate Data Variance
        # We use Stratified K-Fold to ensure class distributions remain consistent across splits.
        splitter = DataSplit(random_state=42, test_size=0.2, k_fold=k_folds)
        x_test, y_test, folds = splitter.k_fold_data_spiting(X, y)

        global_results = []

        for fold_idx in range(k_folds):
            print(f"\n[+] Processing Fold {fold_idx + 1}/{k_folds}")
            fold = folds[fold_idx]

            # Data is already scaled by DataSplit.py inside the fold to prevent data leakage
            x_train, y_train = fold['X_train_fold'].values, fold['y_train_fold'].values
            x_val, y_val     = fold['X_val_fold'].values, fold['y_val_fold'].values

            fold_metrics_accumulator = []

            # 2. Isolate Algorithmic Variance
            # We iterate through predefined seeds to quantify the stability of the swarm convergence.
            for seed in range(seeds_per_fold):

                # We enforce iter_max=100 as per the established literature baseline[cite: 338].
                abc_model = ABC_ELM2(
                    feature_size=x_train.shape[1],
                    hidden_size=hidden_size,
                    activation_function=stable_sigmoid,
                    regularization_lambda=reg_lambda,
                    algo_type=algo_type,
                    random_seed=seed,
                    SN=10, limit=10, iter_max=100
                )

                abc_model.fit(x_train, y_train)
                y_pred = abc_model.predict(x_val)

                # Extract metrics for this specific stochastic trajectory
                metrics = EvaluationMatrix(y_val, y_pred).get_all_metrics()

                # Append seed identifier for granular tracking
                metrics['Fold'] = fold_idx
                metrics['Seed'] = seed
                fold_metrics_accumulator.append(metrics)

            # Aggregate intra-fold metrics to observe swarm stability
            fold_df = pd.DataFrame(fold_metrics_accumulator)

            fold_summary = {
                'Fold': fold_idx,
                'Mean_Accuracy': fold_df['Accuracy'].mean(),
                'StdDev_Accuracy': fold_df['Accuracy'].std(),
                'Mean_F1': fold_df['F1-Score'].mean(),
                'StdDev_F1': fold_df['F1-Score'].std(),
                'Mean_MCC': fold_df['MCC'].mean(),
            }
            global_results.append(fold_summary)
            print(f"    -> Fold {fold_idx} Mean Accuracy: {fold_summary['Mean_Accuracy']:.4f} (±{fold_summary['StdDev_Accuracy']:.4f})")

        # 3. Final Statistical Aggregation
        final_matrix = pd.DataFrame(global_results)
        return final_matrix, pd.DataFrame(fold_metrics_accumulator)

# ==========================================
# Execution Block (Testing the Architecture)
# ==========================================
if __name__ == "__main__":
    # Simulate loading the Gallstone Dataset (Replace with actual pd.read_csv)
    # Using dummy data for structural testing
    print("[*] Generating synthetic dataset for pipeline verification...")
    np.random.seed(42)
    X_dummy = pd.DataFrame(np.random.rand(200, 15))
    y_dummy = pd.Series(np.random.randint(0, 2, 200))

    # Execute the rigorous validation
    start_time = time.time()
    summary_df, granular_df = AcademicEvaluation.rigorous_metaheuristic_validation(
        X=X_dummy,
        y=y_dummy,
        k_folds=5,           # 5 Data splits
        seeds_per_fold=10,   # 10 Swarm initializations per split (Reduce from 50 for initial testing)
        hidden_size=32,
        reg_lambda=1.223,
        algo_type='algo3'
    )

    print(f"\n[!] Global Evaluation Completed in {time.time() - start_time:.2f} seconds.")
    print("\n=== Cross-Fold Statistical Summary ===")
    print(summary_df.to_string(index=False))

[*] Generating synthetic dataset for pipeline verification...
[*] Initializing Rigorous Evaluation: 5-Fold CV with 10 Seeds/Fold.

[+] Processing Fold 1/5
Iteration 1 end : 0.1165s
Iteration 2 end : 0.1344s
Iteration 3 end : 0.1007s
Iteration 4 end : 0.1062s
Iteration 5 end : 0.1305s
Iteration 6 end : 0.1006s
Iteration 7 end : 0.0990s
Iteration 8 end : 0.0967s
Iteration 9 end : 0.1066s
Iteration 10 end : 0.1020s
Iteration 11 end : 0.1193s
Iteration 12 end : 0.0981s
Iteration 13 end : 0.0982s
Iteration 14 end : 0.1169s
Iteration 15 end : 0.1123s
Iteration 16 end : 0.0833s
Iteration 17 end : 0.2231s
Iteration 18 end : 0.0872s
Iteration 19 end : 0.0849s
Iteration 20 end : 0.1365s
Iteration 21 end : 0.0852s
Iteration 22 end : 0.0870s
Iteration 23 end : 0.1001s
Iteration 24 end : 0.1003s
Iteration 25 end : 0.1033s
Iteration 26 end : 0.0920s
Iteration 27 end : 0.0753s
Iteration 28 end : 0.0827s
Iteration 29 end : 0.0867s
Iteration 30 end : 0.0972s
Iteration 31 end : 0.0874s
Iteration 32 end 